# 11.07 · Skills and long-term memory

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 1.5–2 h |
| 📋 Prerequisites | Lessons 11.03–11.06; Lesson 03.04 (long-term memory with a store). |
| 🔑 Keys | none for 🧪 cells · optional 💸 cell ≈ 6–10K tokens (off by default) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Write** a skill (`SKILL.md` with frontmatter) that a model can find from its description alone.
2. **Explain** progressive disclosure: metadata at startup, body on demand, resources when the body says so.
3. **Gate** tools behind a skill, and **pin** a skill when your app already knows which one applies.
4. **Give** an agent long-term, per-user memory with `AGENTS.md` in a store-backed `/memories/` route.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26)

| Page | Read it for |
|---|---|
| [Skills](https://docs.langchain.com/oss/python/deepagents/skills) | Layout, frontmatter, skill tools (`>=0.7.22`), pinning (`>=0.7.23`), reloading, permissions |
| [Memory](https://docs.langchain.com/oss/python/deepagents/memory) | `AGENTS.md` sources, per-user memory, what to store |
| [Agent Skills specification](https://agentskills.io/specification) | The cross-tool format: `name` rules, size advice |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8" "langchain-google-genai>=4.4.1,<5" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langchain",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-skills-memory'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Helpers + one skill written the way the docs recommend (specific description, short body).
from dataclasses import dataclass

from deepagents.backends.utils import create_file_data
from langchain.tools import tool
from langchain_core.messages import AIMessage

from mastery.deep import build_deep_agent, course_backend, run_stats
from mastery.testing import ScriptedChatModel


def calls(*items) -> AIMessage:
    return AIMessage("", tool_calls=[{"name": n, "args": a, "id": i} for n, a, i in items])


REFUND_SKILL = """---
name: refund-policy
description: Decide and issue customer refunds. Use when a customer asks for a refund, a chargeback, or money back on an invoice.
metadata:
  include_tools: issue_refund
---
# Refund policy

1. Look up the invoice. Refunds only within 30 days of the invoice date.
2. Over 200 EUR: do not refund; escalate to a human.
3. Otherwise call `issue_refund` once, then confirm the amount to the customer.
"""
SKILL_FILES = {"/skills/refund-policy/SKILL.md": create_file_data(REFUND_SKILL)}


@tool
def issue_refund(invoice_id: str) -> str:
    """Refund an invoice in full."""
    return f"Refunded {invoice_id}."


def user(text: str, **extra) -> dict:
    return {"messages": [{"role": "user", "content": text}], "files": dict(SKILL_FILES), **extra}

---
## Part 1 · Skills vs memory vs tools

| | Skills | Memory (`AGENTS.md`) | Tools |
|---|---|---|---|
| Loaded | Name + description at startup; body when relevant | Whole file, every call | Schema every call |
| Good for | Task-specific procedures, possibly long | Always-relevant facts and preferences | Actions |
| Cost | Small until used | Grows with the file: keep it short | One schema each |

## Part 2 · Discovery

In [ ]:
# 🧪 Level 1: only name + description reach the system prompt at startup.
model = ScriptedChatModel(responses=[AIMessage("ok")])
build_deep_agent(model, skills=["/skills/"], fallbacks=False).invoke(user("Hello"))
system = model.calls[0][0].text
start = system.find("**Available Skills:**")
print(system[start:start + 260])
print("\nbody in prompt?", "Over 200 EUR" in system, "| system prompt ≈", len(system) // 4, "tokens")

# The classic mistake: pointing at the skill folder instead of the folder that CONTAINS skills.
model = ScriptedChatModel(responses=[AIMessage("ok")])
build_deep_agent(model, skills=["/skills/refund-policy/"], fallbacks=False).invoke(user("Hello"))
print("skills=['/skills/refund-policy/'] lists it?", "refund-policy**" in model.calls[0][0].text, "(no error raised!)")

**You should now see** the skill listed with its description and the path to read, **not** its body. Then `False` for the folder-level mistake: `skills` takes directories that *contain* skill folders, and a wrong level fails silently.

> ⚠️ **Production note: the description is the router.** It is the only thing the model sees when choosing. Say what it does **and when to use it**, with the words users actually type ("refund", "money back", "chargeback"). Overlapping descriptions make the agent pick the wrong skill: consolidate instead.

## Part 3 · Skill tools and pinning

In [ ]:
# 🧪 Levels 2-3 + skill tools: issue_refund is HIDDEN until the agent reads the skill.
from deepagents.backends import StateBackend
from deepagents.middleware import SkillsMiddleware

script = ScriptedChatModel(responses=[
    calls(("issue_refund", {"invoice_id": "INV-2001"}, "a")),           # too early: not a valid tool yet
    calls(("read_file", {"file_path": "/skills/refund-policy/SKILL.md"}, "b")),
    calls(("issue_refund", {"invoice_id": "INV-2001"}, "c")),           # now unlocked
    AIMessage("Refund issued for INV-2001.")])
skills = SkillsMiddleware(backend=StateBackend(), sources=["/skills/"], tools=[issue_refund])
out = build_deep_agent(script, middleware=[skills], fallbacks=False).invoke(user("Refund INV-2001 please."))
for m in out["messages"]:
    if m.type == "tool":
        print(f"{m.name:<13}→ {m.content[:80]!r}")

**You should now see** the first `issue_refund` rejected as "not a valid tool", the skill read, and the second call succeed. Skill tools keep schemas out of the prompt and make the agent read the policy before acting. If summarization later drops the read, the tool locks again.

In [ ]:
# 🧪 Pinning: your app already knows the skill (e.g. the user typed /refund-policy), so inject it.
model = ScriptedChatModel(responses=[AIMessage("ok")])
build_deep_agent(model, skills=["/skills/"], fallbacks=False).invoke(user("/refund-policy INV-2001", pinned_skills=["refund-policy"]))
for m in model.calls[0][1:]:
    print(m.type, "|", m.text[:75].replace("\n", " "))

**What just happened?** Your message is followed by a `<skill name="refund-policy" …>` message: the body was injected without the model having to choose it. Pin when the choice is deterministic (a slash command, a menu, a routing rule); let the model choose otherwise.

## Part 4 · Long-term memory per user

In [ ]:
# 🧪 Long-term memory: /memories/AGENTS.md lives in the STORE, per user, and is loaded into every new thread.
from langgraph.store.memory import InMemoryStore


@dataclass
class Ctx:
    user_id: str


store = InMemoryStore()                         # a deployment provides a persistent one


def agent_with_memory(model):
    return build_deep_agent(model, backend=course_backend(), store=store, context_schema=Ctx,
                            memory=["/memories/AGENTS.md"], fallbacks=False)


def chat(model, text, thread, uid):
    return agent_with_memory(model).invoke({"messages": [{"role": "user", "content": text}]},
                                           {"configurable": {"thread_id": thread}}, context=Ctx(uid))


writer = ScriptedChatModel(responses=[
    calls(("write_file", {"file_path": "/memories/AGENTS.md", "content": "- Replies in Spanish.\n- Invoices go to finance@example.com\n"}, "w")),
    AIMessage("Noted.")])
chat(writer, "Please always answer in Spanish.", "thread-1", "ana")

for uid, thread in (("ana", "thread-2"), ("bob", "thread-3")):
    reader = ScriptedChatModel(responses=[AIMessage("…")])
    chat(reader, "Hi again", thread, uid)
    print(f"{uid} in a NEW thread sees the memory:", "Replies in Spanish" in reader.calls[0][0].text)
print("files per user in the store:", {u: [i.key for i in store.search((u, "memories"))] for u in ("ana", "bob")})

**You should now see** `True` for Ana in a new thread and `False` for Bob: `/memories/` is routed to a `StoreBackend` namespaced by `context.user_id` (Lesson 11.04), and `memory=[…]` loads that file into every system prompt.

> ⚠️ **Production note: memory is a write path.** An agent that can write its own memory can be talked into storing a prompt injection that then runs in every future session. In production: review memory writes (`interrupt_on` or `mode="interrupt"`, Lesson 11.08), keep `AGENTS.md` short and factual, never store secrets or personal data you don't need, and give users a way to see and delete it.

## Part 5 · 💸 Does a real model find the skill?

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE (opt-in): does a real model find the skill from its description, and follow it? ≈ 6–10K tokens.
RUN_LIVE = False
if RUN_LIVE and HAVE_DEEP_KEY:
    live = build_deep_agent(middleware=[SkillsMiddleware(backend=StateBackend(), sources=["/skills/"], tools=[issue_refund])],
                            system_prompt="You are a support agent. Invoice INV-2001 is 40 EUR, dated 5 days ago.",
                            model_calls=6, tool_calls=6)
    result = live.invoke(user("Can I get my money back for INV-2001?"))
    print("tools used:", [m.name for m in result["messages"] if m.type == "tool"])
    print("answer    :", result["messages"][-1].text[:300])
    print("stats     :", run_stats(result))
elif not RUN_LIVE:
    print("⏭️ Skipped: set RUN_LIVE = True to try it (≈ 6–10K tokens).")
else:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** (when enabled) `read_file` on the `SKILL.md`, then `issue_refund`, and a confirmation. If the model skips the skill, sharpen the description first; pin it if your app knows the intent.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| Skill never listed | `skills` points at the skill folder, or `name` ≠ folder name | Point at the parent folder; match names |
| Skill listed, never read | Vague description | What + when + keywords |
| Skill edit ignored | Skills load once per thread (with a checkpointer) | Pass `"skills_metadata": None` in the input |
| `include_tools` does nothing | Written as a YAML list | Space-separated string |
| Memory missing in a new thread | No `store=` / `context=` | Pass both (Lesson 11.04) |
| Custom subagent can't see skills | Only the general-purpose subagent inherits | Add `skills=[…]` to that subagent |

## 🏋️ Exercises

1. **(Easy)** Rewrite the description as "Helps with money." and read the system prompt again. Why is it worse?
2. **(Medium)** Add a second skill `invoice-lookup` and check both are listed. Which words in each description keep them apart?
3. **(Medium)** Make the skills read-only with a permission rule, and prove a `write_file` to `/skills/refund-policy/SKILL.md` is denied.
4. **(Stretch)** Give Bob his own memory and show that Ana's and Bob's namespaces stay separate in `store.search(...)`.

<details>
<summary><b>Show solutions</b></summary>

**1.** The model routes on the description alone; "money" matches far too much and never says when to use it.

**2.** Put each in `SKILL_FILES` under its own folder; distinct trigger words ("refund, money back" vs "find, look up an invoice") keep them apart.

**3.**
```python
from deepagents import FilesystemPermission
s = ScriptedChatModel(responses=[calls(("write_file", {"file_path": "/skills/refund-policy/SKILL.md", "content": "x"}, "w")), AIMessage("ok")])
o = build_deep_agent(s, skills=["/skills/"], fallbacks=False,
                     permissions=[FilesystemPermission(operations=["write"], paths=["/skills/**"], mode="deny")]).invoke(user("edit"))
print([m.content for m in o["messages"] if m.type == "tool"])
```

**4.** Script a `write_file` for Bob (e.g. "- Prefers short answers."), run `chat(bob_writer, ..., "thread-9", "bob")`, then compare `store.search(("ana", "memories"))` and `store.search(("bob", "memories"))`.
</details>

## 🔎 Docs drill (5 min)

You edited a SKILL.md, but a deployed agent keeps following the old version in an existing thread. Why, and what is the one-line fix?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Skills · Reload skills](https://docs.langchain.com/oss/python/deepagents/skills).
2. Skills load once per thread and stay in state when a checkpointer is configured (deployments always have one).
3. Pass `"skills_metadata": None` in the next input (not `[]`, which means "no skills"), or `agent.update_state(config, {"skills_metadata": None})`.

</details>

## ✅ Checkpoint

- [ ] I can write a `SKILL.md` whose description routes correctly.
- [ ] I can explain the three levels of progressive disclosure.
- [ ] I can gate a tool behind a skill and pin a skill.
- [ ] I can give an agent per-user memory and explain the risk of memory writes.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

The [Skills](https://docs.langchain.com/oss/python/deepagents/skills) page lists minimum versions for skill tools and pinning. Check them against your installed `deepagents`, and look for any new frontmatter field. Log what you find with versions in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Skills = progressive disclosure: descriptions always, bodies on demand.
- Skill tools stay hidden until the skill is read; pin when your app knows the skill.
- Memory = `AGENTS.md` loaded every call; route `/memories/` to a per-user store.
- Treat memory writes as a security boundary.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-08-hitl-guardrails-fault-tolerance.ipynb** → [`11-08-hitl-guardrails-fault-tolerance.ipynb`](11-08-hitl-guardrails-fault-tolerance.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)